In [7]:
import hashlib
import random
import time
import string

In [47]:
message = input("Enter a message for hashing: ")

encoded_message = message.encode('utf-8')



md5_hash = hashlib.md5(encoded_message).hexdigest()
sha1_hash = hashlib.sha1(encoded_message).hexdigest()
sha256_hash = hashlib.sha256(encoded_message).hexdigest()
sha512_hash = hashlib.sha512(encoded_message).hexdigest()
sha3_256_hash = hashlib.sha3_256(encoded_message).hexdigest()
sha3_512_hash = hashlib.sha3_512(encoded_message).hexdigest()

print(f"Original message: {message}")
print(f"MD5: {md5_hash}")
print(f"SHA-1: {sha1_hash}")
print(f"SHA-256: {sha256_hash}")
print(f"SHA-512: {sha512_hash}")
print(f"SHA3-256: {sha3_256_hash}")
print(f"SHA3-512: {sha3_512_hash}")


Original message: kwa kwa kwa
MD5: d29ebb1360ea5cefab49054d90d81988
SHA-1: 482883186b6bb28701d287356a5a56d4f5c59b2f
SHA-256: 8a62ed0c20fb662f56531bb9c3c246872a1db8dcfa2c957586edc0eee7b2f598
SHA-512: 1c718ae6ff4ee0b8a1b34004122d22898a7d283cfd3bea9ce209d20298a5390dcef611ac2f6935886f904e0414617f1d31611ed7f6a152c9f08b68ae339f0966
SHA3-256: f057e7342157d3d20a9da8aac0a54eacee93d2e3aff183cfe5c105e8aa08d9f7
SHA3-512: 0f4ebd102db6f54077eb0cdb53aa27f31843be3c24f3378d05b389009e7d2ad2d029393ad86913c5c2727c8c411bc22c1545d0566a89386fb8875189adb3ab30


In [49]:
algorithms = [ hashlib.sha1, hashlib.sha256, hashlib.sha512, hashlib.sha3_256, hashlib.sha3_512, hashlib.md5 ]
results = {}
sample_size = 1000

message_sizes = [10**6, 10**5, 10**4, 10**3]  # 1MB, 100KB, 10KB, 1KB

for message_size in message_sizes:
    random_message = ''.join(random.choices('abcdefghijklmnopqrstuvwxyz0123456789', k=message_size))

    for algo in algorithms:
        results[algo.__name__] = 0

    for _ in range(sample_size):

        encoded_random_message = random_message.encode('utf-8')

        for algo in algorithms:
            start_time = time.perf_counter()
            algo(encoded_random_message).hexdigest()
            end_time = time.perf_counter()
            results[algo.__name__] += (end_time - start_time)


    for key in results:
        results[key] /= sample_size

    print(f"Message length: {message_size} characters")

    for algo_name, duration in sorted(results.items(), key=lambda item: item[1]):
        print(f"{algo_name}: {duration:.6f} seconds")
    print("-" * 50)
    

Message length: 1000000 characters
openssl_sha1: 0.000557 seconds
openssl_sha256: 0.000595 seconds
openssl_sha512: 0.001288 seconds
openssl_md5: 0.001409 seconds
openssl_sha3_256: 0.002320 seconds
openssl_sha3_512: 0.004123 seconds
--------------------------------------------------
Message length: 100000 characters
openssl_sha1: 0.000058 seconds
openssl_sha256: 0.000061 seconds
openssl_sha512: 0.000132 seconds
openssl_md5: 0.000141 seconds
openssl_sha3_256: 0.000234 seconds
openssl_sha3_512: 0.000418 seconds
--------------------------------------------------
Message length: 10000 characters
openssl_sha1: 0.000007 seconds
openssl_sha256: 0.000008 seconds
openssl_sha512: 0.000014 seconds
openssl_md5: 0.000015 seconds
openssl_sha3_256: 0.000025 seconds
openssl_sha3_512: 0.000043 seconds
--------------------------------------------------
Message length: 1000 characters
openssl_sha1: 0.000001 seconds
openssl_sha256: 0.000002 seconds
openssl_md5: 0.000002 seconds
openssl_sha512: 0.000002 sec

In [78]:
def znajdz_kolizje_12bit(algorytm=hashlib.sha256):
    znalezione_prefixy = {}  
    proby = 0

    print(f"Rozpoczynam szukanie kolizji dla {algorytm.__name__}")
    print("-" * 50)

    while True:
        proby += 1
        tekst = ''.join(random.choices(string.ascii_letters + string.digits, k=50))
    
        pelny_hash = algorytm(tekst.encode()).hexdigest()
    
        prefix = pelny_hash[:3]

        if prefix in znalezione_prefixy:
            stary_tekst = znalezione_prefixy[prefix]
            if stary_tekst != tekst:
                print(f"KOLIZJA ZNALEZIONA po {proby} próbach")
                print(f"Tekst 1: {stary_tekst} -> Hash: {algorytm(stary_tekst.encode()).hexdigest()}")
                print(f"Tekst 2: {tekst}       -> Hash: {pelny_hash}")
                print(f"Wspólny prefix (12 bit): {prefix}")
                break
        
        znalezione_prefixy[prefix] = tekst

znajdz_kolizje_12bit()

Rozpoczynam szukanie kolizji dla openssl_sha256
--------------------------------------------------
KOLIZJA ZNALEZIONA po 47 próbach
Tekst 1: wBUQ7ovZsJd3s3xjuQRlfra8yYYRDmYBXPNi27UeHyHC9RHeWn -> Hash: de2137064829cb444cb92ffcff78e6bf7f2ee6e16bec0dbd652211a374162f0a
Tekst 2: 9EwarnMsgtla7H5rvLzeRNcagNCOMGenhYQpOuEJ0xJpZz1Qmt       -> Hash: de2cb5c2f24539ff2f57e6fd1490a2fc3e22c7a29cf7b65e01cca9fdccae5f47
Wspólny prefix (12 bit): de2


In [83]:
def sac_check(algo=hashlib.sha256, test_count=100000):
    print(f"Sprawdzanie SAC dla {algo.__name__}")
 
    sum_diff_percentage = 0

    for _ in range(test_count):

        message = ''.join(random.choices(string.ascii_letters + string.digits, k=50))
        encoded_message = message.encode('utf-8')
        original_hash_hex = algo(encoded_message).hexdigest()
     

        index_to_change = random.randint(0, len(encoded_message) - 1)
        modified_message = bytearray(encoded_message)
        modified_message[index_to_change] ^= 1  
        modified_hash_hex = algo(modified_message).hexdigest()
   
        int_orig = int(original_hash_hex, 16)
        int_mod = int(modified_hash_hex, 16)
        
        diff_bits = (int_orig ^ int_mod).bit_count()
        
        total_bits = len(original_hash_hex) * 4

        diff_percentage = (diff_bits / total_bits) * 100
        sum_diff_percentage += diff_percentage

    average_diff_percentage = sum_diff_percentage / test_count
    print(f"Średnia zmiana bitów: {average_diff_percentage:.2f}%")
    print("-" * 50)

sac_check(hashlib.md5)
sac_check(hashlib.sha256)
sac_check(hashlib.sha512)
sac_check(hashlib.sha3_256)
sac_check(hashlib.sha3_512)
sac_check(hashlib.sha1)


Sprawdzanie SAC dla openssl_md5
Średnia zmiana bitów: 50.03%
--------------------------------------------------
Sprawdzanie SAC dla openssl_sha256
Średnia zmiana bitów: 50.00%
--------------------------------------------------
Sprawdzanie SAC dla openssl_sha512
Średnia zmiana bitów: 50.01%
--------------------------------------------------
Sprawdzanie SAC dla openssl_sha3_256
Średnia zmiana bitów: 50.00%
--------------------------------------------------
Sprawdzanie SAC dla openssl_sha3_512
Średnia zmiana bitów: 50.00%
--------------------------------------------------
Sprawdzanie SAC dla openssl_sha1
Średnia zmiana bitów: 50.01%
--------------------------------------------------
